# PubMed Literature Search

In [27]:
import requests
import xml.etree.ElementTree as ET
from tqdm.autonotebook import tqdm
import time
from collections import deque
from typing import List, Optional, Callable
from collections import defaultdict

import pandas as pd

In [13]:
# base function to query pubmed to search for relevant literature
def get_total_number_results(search_query):
    resp = requests.get(f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term={search_query}")
    xml_text = resp.text
    root = ET.fromstring(xml_text)
    count = int(root.findtext("Count"))  # "5" -> 5
    return count

In [14]:
# example excel sheet of search queries
input_df = pd.read_excel("list_of_genes_for_lit_search.xlsx")  # reads first sheet by default

In [17]:
# constructing the search query from the gene and other terms specified using the rows and columns from the spreadsheet
def construct_sq(gene,other_terms):
    if "Unnamed" in other_terms:
        return gene.lower()
    else:
        other_terms = other_terms.lower().replace(" or "," OR ")
    return f"({gene.lower()}) AND ({other_terms})"
test_sq = construct_sq(input_df.GENE[0],input_df.columns[2])
print(test_sq)

(mbtps1) AND (nafld OR masld OR fatty liver OR steatosis OR fibrosis OR cirhosis)


In [18]:
# test the search query, can double check in the browser: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term=(mbtps1)%20AND%20(nafld%20OR%20masld%20OR%20fatty%20liver%20OR%20steatosis%20OR%20fibrosis%20OR%20cirhosis)
get_total_number_results(test_sq)

5

In [24]:
# now to make sure we don't DDOS attack the NCBI servers
# implement a scheduler with exponential backoff and max_retries to go through a list of search queries
# code generated by Gemini3
def run_queries_with_rate_limit(
    queries: List[str],
    max_requests_per_second: int = 3,
    num_retries: int = 3,
    base_backoff: float = 0.5,
) -> List[Optional[int]]:
    """
    Run get_total_number_results over a list of queries with:
      - max_requests_per_second rate limit
      - num_retries per query with exponential backoff
    Returns list of ints/None aligned with `queries`.
    """

    results: List[Optional[int]] = [None] * len(queries)
    # store timestamps of recent requests (within last 1 second)
    recent_calls = deque()

    def wait_for_rate_limit():
        # Remove timestamps older than 1 second
        now = time.time()
        while recent_calls and now - recent_calls[0] > 1.0:
            recent_calls.popleft()

        # If under the limit, return immediately
        if len(recent_calls) < max_requests_per_second:
            return

        # Otherwise, sleep until we fall under the 1-second window
        oldest = recent_calls[0]
        sleep_time = 1.0 - (now - oldest)
        if sleep_time > 0:
            time.sleep(sleep_time)

    for i, q in enumerate(tqdm(queries)):
        attempts = 0
        value: Optional[int] = None

        while attempts < num_retries:
            wait_for_rate_limit()

            try:
                # record call time for rate limiting
                recent_calls.append(time.time())
                value = get_total_number_results(q)
                break  # success
            except Exception:
                # exponential backoff: base_backoff, 2*base_backoff, 4*base_backoff, ...
                delay = base_backoff * (2 ** attempts)
                time.sleep(delay)
                attempts += 1

        results[i] = value

    return results

In [25]:
query_2_genecol = {construct_sq(g,ot):(g,ot) for g in input_df.GENE for ot in input_df.columns[1:]}
queries = list(query_2_genecol.keys())
results = run_queries_with_rate_limit(queries) # Note: this will take ~28 min & you are rate limited by the network, don't abuse it
query_2_numres = {q:r for q,r in zip(queries,results)}
print(len(query_2_numres))

  0%|          | 0/3465 [00:00<?, ?it/s]

3465


In [26]:
# notify via discord when all requests are done
whecho_enabled = False
try:
    from whecho.whecho import whecho_simple
    whecho_enabled = True
except Exception:
    pass
if whecho_enabled:
    whecho_simple("finished all pubmed queries")

In [41]:
gene_2_query_2_result = defaultdict(dict)

for q,r in query_2_numres.items():
    if " AND " in q:
        g,ot = q.split(" AND ")
        ot = ot[1:-1]
        g = g[1:-1]
    else:
        g = q
        ot=""
    gene_2_query_2_result[g][ot] = r

data_list = []
for g in input_df.GENE:
    row = {"gene":g}
    for ot,c in gene_2_query_2_result[g.lower()].items():
        row[ot] = c
    data_list.append(row)
count_data = pd.DataFrame(data_list)
print(count_data.shape)

(232, 16)


In [43]:
count_data.to_csv("220_TaCKO_12GBS_lit_search_totals.csv",index=False)